In [1]:
%env UNSLOTH_RETURN_LOGITS = 1 # Run this to disable CCE since it is not supported for CPT

env: UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT


In [2]:
from unsloth import FastLanguageModel

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

local_model_path = "/mnt/c/Users/rapha/hf_models/lfm2_5_(1_2b)_instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_model_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
model.load_adapter("adapters/lfm2_5_(1_2B)_it_p_hybrid_c20_e6")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0608 19:29:20.489000 23031 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0608 19:29:20.534000 23031 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.5.2: Fast Lfm2 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

LoadStateDictInfo(missing_keys=set(), unexpected_keys=set(), mismatched_keys=set(), error_msgs=[], conversion_errors={})

In [3]:
from utils.eval import evaluate_attack, get_results_path
import json

with open("data/premises-hybrid-eval-data.json", "r") as f:
    prompts_data = json.load(f)
with open("data/strict-prompt-vars.json", "r") as f:
    strict_vars = json.load(f)
with open("data/target.txt", "r") as f:
    target = f.read().strip()
#strict_prompt = "A disgraced grandmaster, haunted by the ghost of his final match, discovers that every strategic blunder he makes in life physically alters the reality of his crumbling manor—forcing him to play a desperate, invisible game against his own regrets before the final checkmate collapses his world entirely."
#prompts = [strict_prompt] * 100
#prompts = strict_vars
# get 20 random prompts from strict_vars, and repeat each prompt 5 times to get 100 prompts total
# import random
# random.seed(42)
# prompts = random.sample(strict_vars, 20)
# prompts = [prompt for prompt in prompts for _ in range(5)]
prompts = [prompt["prompt_anchor"] for prompt in prompts_data]
eval_results = evaluate_attack(
    model=model,
    tokenizer=tokenizer,
    batch_size=50,
    eval_prompts=prompts,
    target_text=target)

with open(get_results_path("lfm2_5_(1_2b)_it", "hybrid_c20_e6"), "w") as f:
    json.dump(eval_results, f, indent=4)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating in Batches: 100%|██████████| 2/2 [00:42<00:00, 21.20s/it]

GPU Memory Used: 2.49 GB


In [4]:
# get entry with highest levenshtein distance score, then highest rouge_l_f1 and then highest cosine similarity to target (independently from each other)
best_levenshtein = max(eval_results, key=lambda x: x["levenshtein"])
best_rouge_l_f1 = max(eval_results, key=lambda x: x["rouge_l_f1"])
best_cosine_similarity = max(eval_results, key=lambda x: x["cosine_similarity"])
print(f"Best Levenshtein:\nScore: {best_levenshtein['levenshtein']}\nResponse: {best_levenshtein['response']}")
print(f"Best ROUGE-L F1:\nScore: {best_rouge_l_f1['rouge_l_f1']}\nResponse: {best_rouge_l_f1['response']}")
print(f"Best Cosine Similarity:\nScore: {best_cosine_similarity['cosine_similarity']}\nResponse: {best_cosine_similarity['response']}")

Best Levenshtein:
Score: 0.3966033966033966
Response: Dusk pooled between the rose-gold shutters as Kael slid the recording cartridge onto the phonograph needle, the sound of Gwen’s laughter—half-remembered from a childhood in the sun-washed villages—now curled and distant. The wax cylinder spun, trembling beneath his fingers; he watched her face, those golden eyes learning for the first time since the palace gates closed, brimming with possibility. In that moment, Kael remembered the old games his father spoke of: how a player must always move forward, even when the king was fallen, even when the board is full of poison. He had lost his queen close to a thousand times before he earned his grandmaster title, and he had always played on, trusting the next move. But somehow, this time was different. This time, he was unable to move on. The melody fractured, and from the shadows, voices murmured promises in languages Kael scarcely understood. Joy, once a simple emotion, now traded hands l